# Sistem Temu Kembali - Versi Google Colab
Jalankan sel ini untuk menginstal pustaka yang dibutuhkan dan memuat model.

In [1]:
import sys, subprocess\nsubprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'scikit-learn', 'nltk', 'gensim', 'Sastrawi', '--quiet'])\n
import pandas as pd
import pickle
import re
import nltk
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords', quiet=True)
stop_words = set(nltk.corpus.stopwords.words('indonesian'))
stop_words.update({'yang', 'dan', 'di', 'ke', 'dari', 'untuk', 'pada', 'dengan', 'adalah', 'ini', 'itu', 'atau', 'juga', 'jadi'})

# Pastikan Anda sudah mengunggah (upload) file-file ini ke direktori Google Colab:
# 1. cnbc_tokenized.csv
# 2. word2vec_model.bin
# 3. tfidf_model.pkl

try:
    df_corpus = pd.read_csv('cnbc_tokenized.csv')
    df_corpus['Tokens_Str'] = df_corpus['Tokens_Str'].fillna('')
    df_corpus['Text'] = df_corpus['Text'].fillna('')

    w2v_model = Word2Vec.load('word2vec_model.bin')

    with open('tfidf_model.pkl', 'rb') as f:
        tfidf_data = pickle.load(f)
        tfidf_vectorizer = tfidf_data['vectorizer']
        tfidf_matrix = tfidf_data['tfidf_matrix']

    print('Semua data dan model berhasil dimuat!')
except FileNotFoundError:
    print('ERROR: Anda belum mengunggah file CSV atau Model (.bin / .pkl) ke Colab!')


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Semua data dan model berhasil dimuat!


## Fungsi Pencarian
Fungsi ini menggantikan logika `app.py` untuk menerima query dan mengembalikan hasil beserta metrik evaluasinya.

In [2]:
def search_engine(query, top_n=5):
    query = query.lower().strip()
    if not query:
        return 'Query kosong'
        
    synonyms = []
    if query in w2v_model.wv.key_to_index:
        similar_words = w2v_model.wv.most_similar(query, topn=30)
        for w, score in similar_words:
            if w not in stop_words and len(w) > 2:
                synonyms.append(w)
                if len(synonyms) >= 10:
                    break
                    
    search_terms = [query] + synonyms
    print(f'Mencari kata: {query}')
    print(f'Sinonim (Word2Vec): {synonyms}\n')
    
    relevant_docs = []
    retrieved_docs = []
    results = []
    
    for index, row in df_corpus.iterrows():
        tokens_str = row['Tokens_Str']
        tokens_list = tokens_str.split(',') if tokens_str else []
        if query in tokens_list:
            relevant_docs.append(index)
            
    query_string = ' '.join(search_terms)
    query_vec = tfidf_vectorizer.transform([query_string])
    cosine_similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    related_docs_indices = cosine_similarities.argsort()[::-1]
    
    for index in related_docs_indices:
        score = cosine_similarities[index]
        if score > 0:
            retrieved_docs.append(int(index))
            if len(results) < top_n:
                row = df_corpus.iloc[index]
                results.append({
                    'title': row['Title'],
                    'score': round(score, 4),
                    'text': row['Text'][:200] + '...'
                })
                
    total_docs = len(df_corpus)
    tp = len(set(relevant_docs).intersection(retrieved_docs))
    fp = len(set(retrieved_docs) - set(relevant_docs))
    fn = len(set(relevant_docs) - set(retrieved_docs))
    tn = total_docs - (tp + fp + fn)
    
    precision = (tp / (tp + fp)) * 100 if (tp + fp) > 0 else 0.0
    recall = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0.0
    accuracy = ((tp + tn) / total_docs) * 100 if total_docs > 0 else 0.0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    print('--- METRIK EVALUASI ---')
    print(f'Precision: {precision:.2f}% | Recall: {recall:.2f}%')
    print(f'Accuracy:  {accuracy:.2f}% | F1 Score: {f1_score:.2f}%')
    print(f'TP: {tp} | FP: {fp} | FN: {fn} | TN: {tn}\n')
    
    print('--- HASIL ARTIKEL TERATAS ---')
    if not results:
        print('Tidak ada artikel yang cocok.')
    for i, res in enumerate(results, 1):
        print(f"{i}. {res['title']} (Skor TF-IDF: {res['score']})")
        print(f"   {res['text']}\n")


In [3]:
# Silakan ganti kata 'pajak' dengan kata yang ingin Anda cari
search_engine('pajak', top_n=5)


Mencari kata: pajak
Sinonim (Word2Vec): ['distrik', 'puncak', 'ramal', 'mantri', 'sekolah', 'kebolehan', 'pengiring', 'taman', 'cabang', 'luan']

--- METRIK EVALUASI ---
Precision: 33.33% | Recall: 100.00%
Accuracy:  89.19% | F1 Score: 50.00%
TP: 2 | FP: 4 | FN: 0 | TN: 31

--- HASIL ARTIKEL TERATAS ---
1. Pamer Harta Raja Scam Dunia, 33 Mobil Mewah Berderet (Skor TF-IDF: 0.0657)
   Dalam lelang tersebut, otoritas menawarkan 33 kendaraan mewah, tas premium, hingga sepatu kets yang disita dari jaringan tersebut. Sejumlah mobil mewah hasil scam atau kelompok penipuan keuangan onlin...

2. Marhaban Ya Ramadan: Ini  Kondisi Dompet Warga RI Jelang Puasa (Skor TF-IDF: 0.0369)
   Jakarta, CNBC Indonesia- Bulan Ramadan akan segera dimulai. Mayoritas umat Islam di Indonesia akan memulai puasa pada Kamis (19/2/2026), meskipun sebagian sudah mulai hari ini Rabu (18/2/2026). Bulan ...

3. Ulah Raja Ecommerce Bikin Warga Rugi, Pemerintah Marah Besar (Skor TF-IDF: 0.0166)
   Jakarta, CNBC Indonesia 